# 📰 Fake News Detection using NLP & Machine Learning
**Dataset:** ISOT Fake and Real News (`True.csv` + `Fake.csv`)  
**Goal:** Classify a news article as **REAL (1)** or **FAKE (0)** from its text.

**Pipeline:** Load → Clean → EDA (plots) → TF-IDF → Train models → Evaluate → Predict on new text

## 1. Setup

In [ ]:
import re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 2. Load the data in Google Colab
Pick **ONE** of the three options below (run only one cell).

**Option A – Upload directly (quickest):** click the 📁 folder icon on the left → upload `True.csv` and `Fake.csv` → they land in `/content/`.  
(Uploaded files vanish when the runtime resets.)

**Option B – Google Drive (persistent):** put the CSVs in `MyDrive/fake_news/`.

**Option C – From your GitHub repo:** large files (>100 MB) don't fit normal GitHub; these two are ~55–65 MB each so they do, but a Kaggle download is cleaner.

In [ ]:
# ---- OPTION A: files uploaded to /content ----
TRUE_PATH = "/content/True.csv"
FAKE_PATH = "/content/Fake.csv"

In [ ]:
# ---- OPTION B: Google Drive ----
# from google.colab import drive
# drive.mount('/content/drive')
# TRUE_PATH = "/content/drive/MyDrive/fake_news/True.csv"
# FAKE_PATH = "/content/drive/MyDrive/fake_news/Fake.csv"

In [ ]:
# ---- OPTION C: clone your GitHub repo (replace the URL) ----
# !git clone https://github.com/<your-username>/fake-news-detection.git
# TRUE_PATH = "/content/fake-news-detection/data/True.csv"
# FAKE_PATH = "/content/fake-news-detection/data/Fake.csv"

In [ ]:
true_df = pd.read_csv(TRUE_PATH)
fake_df = pd.read_csv(FAKE_PATH)
print("True:", true_df.shape, "| Fake:", fake_df.shape)
true_df.head()

## 3. Cleaning (important!)
Two **data-leakage traps** exist in this dataset and inflate accuracy to ~99.9% if ignored:
1. Almost every real article begins with a location tag + **"(Reuters) -"**; very few fake ones do.
2. The `subject` column is disjoint: `politicsNews`/`worldnews` appear only in real news, while `News`, `politics`, `left-news`… appear only in fake news.

So we remove the Reuters tag from the text and **do not use `subject` as a feature**.

In [ ]:
true_df["label"] = 1   # REAL
fake_df["label"] = 0   # FAKE
df = pd.concat([true_df, fake_df], ignore_index=True)

# 1) drop duplicates and empty texts
before = len(df)
df = df.drop_duplicates(subset=["title", "text"])
df = df[df["text"].str.strip().str.len() > 0]
print(f"Removed {before - len(df)} duplicate/empty rows -> {len(df)} left")

# 2) strip the 'CITY (Reuters) -' source tag (leakage)
def strip_source_tag(t):
    t = re.sub(r"^.{0,80}?\(Reuters\)\s*-\s*", "", t)
    return re.sub(r"Reuters", "", t, flags=re.I)

def clean_text(t):
    t = strip_source_tag(t)
    t = re.sub(r"http\S+|www\.\S+", " ", t)      # urls
    t = re.sub(r"[^A-Za-z\s]", " ", t)           # punctuation / numbers
    return re.sub(r"\s+", " ", t).strip().lower()

df["content"] = (df["title"] + " " + df["text"]).apply(clean_text)
df["text_len"] = df["text"].str.split().str.len()
df["date_parsed"] = pd.to_datetime(df["date"].str.strip(), errors="coerce")
df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
df[["title", "label", "subject", "text_len"]].head()

## 4. Exploratory Data Analysis (visualisations)

### 4.1 Class balance

In [ ]:
fig, ax = plt.subplots(figsize=(5,4))
sns.countplot(x=df["label"].map({0:"Fake", 1:"Real"}), palette=["#e74c3c","#2ecc71"], ax=ax)
ax.set_title("Class distribution"); ax.set_xlabel("")
for p in ax.patches: ax.annotate(int(p.get_height()), (p.get_x()+p.get_width()/2, p.get_height()), ha="center", va="bottom")
plt.tight_layout(); plt.savefig("class_balance.png", dpi=150); plt.show()

### 4.2 Subject by class (shows the leakage)

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
sns.countplot(data=df, y="subject", hue=df["label"].map({0:"Fake",1:"Real"}),
              palette=["#e74c3c","#2ecc71"], ax=ax)
ax.set_title("Subject categories never overlap between Fake and Real")
plt.tight_layout(); plt.savefig("subject_by_class.png", dpi=150); plt.show()

### 4.3 Article length

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
sns.histplot(data=df[df.text_len < 2000], x="text_len", hue=df["label"].map({0:"Fake",1:"Real"}),
             bins=60, palette=["#e74c3c","#2ecc71"], element="step", ax=ax)
ax.set_title("Word count per article (capped at 2000)"); ax.set_xlabel("words")
plt.tight_layout(); plt.savefig("text_length.png", dpi=150); plt.show()
print(df.groupby("label")["text_len"].describe()[["mean","50%","max"]])

### 4.4 Articles over time

In [ ]:
ts = df.dropna(subset=["date_parsed"])
ts = ts.assign(month=ts["date_parsed"].dt.to_period("M").dt.to_timestamp())
monthly = ts.groupby(["month", "label"]).size().unstack(fill_value=0).rename(columns={0:"Fake",1:"Real"})
ax = monthly.plot(figsize=(9,4), color=["#e74c3c","#2ecc71"])
ax.set_title("Articles per month (rows with a valid date only)"); ax.set_ylabel("articles")
plt.tight_layout(); plt.savefig("timeline.png", dpi=150); plt.show()
print(f"Fake rows with unparseable dates: {(df[df.label==0]['date_parsed'].isna()).sum()}")

### 4.5 Most frequent words

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
def top_words(texts, n=15):
    cv = CountVectorizer(stop_words="english", max_features=5000)
    X = cv.fit_transform(texts)
    freq = np.asarray(X.sum(axis=0)).ravel()
    return pd.Series(freq, index=cv.get_feature_names_out()).nlargest(n)

fig, axes = plt.subplots(1, 2, figsize=(12,5))
top_words(df[df.label==0]["content"]).sort_values().plot.barh(ax=axes[0], color="#e74c3c")
axes[0].set_title("Top words – FAKE")
top_words(df[df.label==1]["content"]).sort_values().plot.barh(ax=axes[1], color="#2ecc71")
axes[1].set_title("Top words – REAL")
plt.tight_layout(); plt.savefig("top_words.png", dpi=150); plt.show()

### 4.6 Word clouds (optional, `wordcloud` is preinstalled in Colab)

In [ ]:
from wordcloud import WordCloud
fig, axes = plt.subplots(1, 2, figsize=(14,5))
for ax, lab, name in zip(axes, [0,1], ["FAKE","REAL"]):
    wc = WordCloud(width=700, height=400, background_color="white", stopwords=None,
                   max_words=150).generate(" ".join(df[df.label==lab]["content"].sample(3000, random_state=1)))
    ax.imshow(wc); ax.axis("off"); ax.set_title(name)
plt.tight_layout(); plt.savefig("wordclouds.png", dpi=150); plt.show()

## 5. Train / test split & TF-IDF

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["content"], df["label"], test_size=0.2, stratify=df["label"], random_state=RANDOM_STATE)

tfidf = lambda: TfidfVectorizer(stop_words="english", max_df=0.7, min_df=5,
                                ngram_range=(1,2), max_features=100_000, sublinear_tf=True)
print(len(X_train), "train /", len(X_test), "test")

## 6. Train and compare models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, C=5),
    "Linear SVM":          LinearSVC(C=1),
    "Naive Bayes":         MultinomialNB(alpha=0.1),
    "Random Forest":       RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=RANDOM_STATE),
}
pipes, results = {}, []
for name, clf in models.items():
    pipe = Pipeline([("tfidf", tfidf()), ("clf", clf)]).fit(X_train, y_train)
    acc = accuracy_score(y_test, pipe.predict(X_test))
    pipes[name] = pipe; results.append((name, acc)); print(f"{name:20s} accuracy = {acc:.4f}")

res = pd.DataFrame(results, columns=["model","accuracy"]).sort_values("accuracy")
ax = res.plot.barh(x="model", y="accuracy", legend=False, figsize=(7,3.5), color="#3498db")
ax.set_xlim(res.accuracy.min()-0.01, 1.0); ax.set_title("Model comparison (test accuracy)")
plt.tight_layout(); plt.savefig("model_comparison.png", dpi=150); plt.show()

## 7. Evaluate the best model

In [ ]:
best_name = max(results, key=lambda r: r[1])[0]
best = pipes[best_name]
pred = best.predict(X_test)
print("Best model:", best_name)
print(classification_report(y_test, pred, target_names=["Fake","Real"], digits=4))

fig, ax = plt.subplots(figsize=(4.5,4))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=["Fake","Real"]).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion matrix – {best_name}")
plt.tight_layout(); plt.savefig("confusion_matrix.png", dpi=150); plt.show()

### Which words drive the decision? (Logistic Regression coefficients)

In [ ]:
lr = pipes["Logistic Regression"]
names = lr["tfidf"].get_feature_names_out()
coef = lr["clf"].coef_[0]
top_real = pd.Series(coef, index=names).nlargest(15)
top_fake = pd.Series(coef, index=names).nsmallest(15)

fig, axes = plt.subplots(1, 2, figsize=(12,5))
top_fake.sort_values(ascending=False).plot.barh(ax=axes[0], color="#e74c3c"); axes[0].set_title("Words pushing → FAKE")
top_real.sort_values().plot.barh(ax=axes[1], color="#2ecc71"); axes[1].set_title("Words pushing → REAL")
plt.tight_layout(); plt.savefig("feature_importance.png", dpi=150); plt.show()

## 8. Predict on a new article + save the model

In [ ]:
def predict_news(text, model=best):
    p = model.predict([clean_text(text)])[0]
    return "REAL ✅" if p == 1 else "FAKE ❌"

print(predict_news("The Senate passed a budget bill on Tuesday after weeks of negotiation between both parties."))
print(predict_news("BREAKING!!! Secret documents PROVE the media is hiding the truth from you, share before it is deleted!"))

In [ ]:
import joblib
joblib.dump(best, "fake_news_model.joblib")
# from google.colab import files; files.download("fake_news_model.joblib")

## 9. Limitations
- Dataset covers **2015–2018, mostly US politics** → the model learns *style & vocabulary of these sources*, not factual truth.
- Real = Reuters only; Fake = flagged websites. A classifier of **writing style**, not a fact-checker.
- Check generalisation on another dataset (e.g. LIAR, FakeNewsNet) before any real-world claims.